In [1]:
from datasets import load_dataset
import random

dataset = load_dataset("nyu-mll/glue", "sst2")

# SST-2 label mapping: 0 = negative, 1 = positive
LABEL_TO_TEXT = {0: "negative", 1: "positive"}

# Few-shot pool: sampled from TRAIN only. Keep dev and test strictly separate.
train_sentences = dataset["train"]["sentence"]
train_labels = dataset["train"]["label"]

# Reserve validation for prompt selection and test for final evaluation.
# We won't touch these yet — just confirm they're loaded.
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

Train size: 67349
Validation size: 872
Test size: 1821


In [2]:
def sample_shots(k, seed=42):
    """
    Draw k balanced examples from the training set.
    For k=0 returns an empty list.
    For k>0 returns roughly half positive, half negative, shuffled.
    """
    if k == 0:
        return []

    rng = random.Random(seed)

    # Indices of each class in train
    pos_idx = [i for i, y in enumerate(train_labels) if y == 1]
    neg_idx = [i for i, y in enumerate(train_labels) if y == 0]

    n_pos = k // 2
    n_neg = k - n_pos  # if k is odd, one extra negative; fine either way

    chosen = rng.sample(pos_idx, n_pos) + rng.sample(neg_idx, n_neg)
    rng.shuffle(chosen)

    return [
        {"sentence": train_sentences[i], "label": train_labels[i]}
        for i in chosen
    ]

In [3]:
INSTRUCTION = (
    "Classify the sentiment of the movie review as either positive or negative."
)

def build_prompt(test_sentence, shots):
    """
    Build a prompt string with the instruction, optional few-shot examples,
    and the test sentence awaiting a label.
    """
    parts = [INSTRUCTION, ""]

    for ex in shots:
        parts.append(f'Review: "{ex["sentence"]}"')
        parts.append(f"Sentiment: {LABEL_TO_TEXT[ex['label']]}")
        parts.append("")

    parts.append(f'Review: "{test_sentence}"')
    parts.append("Sentiment:")

    return "\n".join(parts)

In [4]:
# Pick one validation example to inspect (we're NOT evaluating yet)
example_sentence = dataset["validation"]["sentence"][0]

for k in [0, 3, 10]:
    shots = sample_shots(k, seed=42)
    prompt = build_prompt(example_sentence, shots)
    print(f"===== k = {k} =====")
    print(prompt)
    print()

===== k = 0 =====
Classify the sentiment of the movie review as either positive or negative.

Review: "it 's a charming and often affecting journey . "
Sentiment:

===== k = 3 =====
Classify the sentiment of the movie review as either positive or negative.

Review: "the one not-so-small problem with expecting is that the entire exercise has no real point . "
Sentiment: negative

Review: "at every opportunity to do something clever "
Sentiment: positive

Review: "tear your eyes away "
Sentiment: negative

Review: "it 's a charming and often affecting journey . "
Sentiment:

===== k = 10 =====
Classify the sentiment of the movie review as either positive or negative.

Review: "fascinating and playful "
Sentiment: positive

Review: "-- and especially williams , an american actress who becomes fully english "
Sentiment: positive

Review: "each other so intensely , but with restraint "
Sentiment: positive

Review: "bind us "
Sentiment: negative

Review: "you 're an absolute raving star wars